# 전세대출 추천 스코어링 파이프라인

**실행 파일 (4개 업로드)**
| 파일 | 설명 |
|------|------|
| `jeonse_customer_profiles_500.csv` | 고객 개인·재무 정보 (500행) |
| `jeonse_property_gate_500.csv` | 임차주택·계약 정보 (500행) |
| `merge_jeonse_product_catalog.csv` | 정부지원 상품 A1~A5 |
| `fss_jeonse_merged_20260630_1.csv` | 금감원 API 실제 상품 (은행/저축은행/보험) |

**실행 순서**: Cell 0 → 1 → 2 → 3 → 4 → 5 → 6 → 7

In [ ]:
# -*- coding: utf-8 -*-
# ============================================================
# [전세대출 추천 스코어링 실습]  — Colab
# 적금 실습과 같은 흐름: read_csv → minmax/reverse → Gate → score → top → action/reason → 저장
# 차이점: 적금은 '고객 1명=1행'이지만, 전세는 '고객 x 상품' 조합을 평가한다.
# ============================================================


# ===== [Cell 0] (준비) 파일 업로드 + 통과 잘 되는 고객 증식 =====
import glob
import numpy as np
import pandas as pd

TOP_N          = 5      # 고객당 추천 상품 수
N_NEW          = 0      # 통과율 높은 신규 고객 수 (0이면 증식 안 함)
SEED           = 42

# Colab이면 파일 업로드(로컬에 이미 있으면 건너뜀)
if not all(glob.glob(p) for p in ["jeonse_customer_profiles*.csv", "jeonse_property_gate*.csv",
                                   "*product_catalog*.csv", "fss*merged*.csv"]):
    try:
        from google.colab import files
        print("CSV 4개 업로드: customer_profiles / property_gate / product_catalog / fss_...merged")
        files.upload()
    except ImportError:
        pass

def _find(pats, exclude=None):
    for pat in pats:
        for f in sorted(glob.glob(pat)):
            if not (exclude and exclude in f):
                return f
# N_NEW>0 이면 원본에서 증식, N_NEW==0 이면 이미 만들어둔 확장(expanded) 파일을 그대로 입력으로 사용
if N_NEW > 0:
    P_PROF = _find(["jeonse_customer_profiles_500.csv"], exclude="expanded")
    P_GATE = _find(["jeonse_property_gate_500.csv"], exclude="expanded")
else:
    P_PROF = _find(["*profiles_expanded.csv", "jeonse_customer_profiles_500.csv"])
    P_GATE = _find(["*gate_expanded.csv", "jeonse_property_gate_500.csv"])
P_CAT  = _find(["merge_jeonse_product_catalog.csv", "*product_catalog*.csv"])
P_FSS  = _find(["fss_jeonse_merged_20260630_1.csv", "fss*merged*.csv", "*merged*.csv"])

# 경로 확인 — None이면 파일 없음 → 업로드 필요
_missing = {k: v for k, v in {"P_PROF": P_PROF, "P_GATE": P_GATE, "P_CAT": P_CAT, "P_FSS": P_FSS}.items() if v is None}
if _missing:
    raise FileNotFoundError(
        f"파일을 찾을 수 없습니다: {list(_missing.keys())}\n"
        f"현재 디렉토리 CSV 목록: {sorted(glob.glob('*.csv'))}\n"
        "→ 위 4개 파일을 Colab에 업로드한 뒤 Cell 0부터 다시 실행하세요."
    )
print(f"P_PROF={P_PROF}\nP_GATE={P_GATE}\nP_CAT ={P_CAT}\nP_FSS ={P_FSS}")

# 다양한 세그먼트(아키타입)로 고객 생성 → 청년/신혼/신생아/저신용/고소득/한도부족/상환위험 골고루
def grow_customers(prof0, gate0, n, seed):
    rng = np.random.default_rng(seed)
    s = int(prof0["user_id"].str.replace("U", "").astype(int).max()) + 1
    CAP=[("서울","강남구"),("서울","송파구"),("서울","마포구"),("서울","노원구"),("경기","성남시"),
         ("경기","고양시"),("경기","화성시"),("인천","연수구"),("인천","부평구")]
    NON=[("부산","해운대구"),("부산","부산진구"),("대구","수성구"),("대전","서구"),("광주","북구"),
         ("울산","남구"),("충남","천안시"),("경남","창원시"),("강원","원주시"),("전북","전주시"),("경북","포항시")]
    PT=["아파트","오피스텔","연립","다세대","단독"]; RP=["원리금균등","원금균등","만기일시"]; IR=["고정","변동","혼합"]
    # 아키타입별 비중 → 개수만큼 라벨 생성 후 섞기
    ARCH=[("청년저소득",.16),("신혼",.16),("신생아",.10),("저신용",.12),
          ("일반무주택",.14),("고소득고보증금",.12),("한도부족",.10),("상환위험",.10)]
    labels=[]
    for nm,fr in ARCH: labels += [nm]*round(fr*n)
    labels=(labels+["일반무주택"]*n)[:n]; rng.shuffle(labels)
    pr=[]; gr=[]
    for i,a in enumerate(labels):
        uid=f"U{s+i:04d}"
        homeless="Y"; head="Y"; mar="미혼"; my=0; nb="N"; dup="N"; info="Y"
        dsr=round(float(rng.uniform(5,32)),1); size="med"; cap=rng.random()<0.5
        emp=rng.choice(["정규직","정규직","계약직","자영업","프리랜서"])
        if a=="청년저소득":                                                   # → A2·A5
            age=int(rng.integers(19,35)); ann=int(rng.integers(1200,3200)); sp=0; credit=int(rng.integers(680,860)); size="small"
        elif a=="신혼":                                                       # → A3
            age=int(rng.integers(26,42)); mar="기혼"; my=int(rng.integers(0,8)); sp=int(rng.integers(1000,4500))
            ann=int(rng.integers(2500,5500)); credit=int(rng.integers(690,920))
        elif a=="신생아":                                                     # → A4
            age=int(rng.integers(28,44)); mar="기혼"; my=int(rng.integers(0,4)); nb="Y"; sp=int(rng.integers(1500,6000))
            ann=int(rng.integers(4000,10000)); credit=int(rng.integers(700,930))
        elif a=="저신용":                                                     # 은행(650)탈락·기금(600)통과
            age=int(rng.integers(24,55)); credit=int(rng.integers(600,650)); ann=int(rng.integers(2000,4800))
            sp=0 if rng.random()<0.6 else int(rng.integers(500,2500))
            if sp>0: mar="기혼"; my=int(rng.integers(1,10))
            dsr=round(float(rng.uniform(8,28)),1)
        elif a=="일반무주택":                                                 # → A1 + 은행
            age=int(rng.integers(30,58)); ann=int(rng.integers(3000,5000)); sp=0 if rng.random()<0.5 else int(rng.integers(500,2000))
            if sp>0: mar="기혼"; my=int(rng.integers(3,20))
            credit=int(rng.integers(700,940))
        elif a=="고소득고보증금":                                             # → 은행(고한도), 무주택 아닌 경우 섞임
            age=int(rng.integers(33,60)); ann=int(rng.integers(7000,14000)); sp=int(rng.integers(0,6000))
            if sp>0: mar="기혼"; my=int(rng.integers(2,18))
            credit=int(rng.integers(740,960)); size="large"; homeless="Y" if rng.random()<0.55 else "N"
        elif a=="한도부족":                                                   # 큰 보증금 → 한도부족형
            age=int(rng.integers(30,55)); ann=int(rng.integers(4000,8000)); sp=int(rng.integers(0,4000))
            if sp>0: mar="기혼"; my=int(rng.integers(2,15))
            credit=int(rng.integers(700,930)); size="large"; cap=True
        else:                                                                 # 상환위험 (DSR 35~40, <=40 유지)
            age=int(rng.integers(28,55)); ann=int(rng.integers(3000,6000)); sp=int(rng.integers(0,3000))
            if sp>0: mar="기혼"; my=int(rng.integers(1,15))
            credit=int(rng.integers(650,780)); dsr=round(float(rng.uniform(35,39.5)),1)
        # 다양화: 일부는 비세대주/유주택/중복대출 → 은행전용 경로나 일부 탈락(현실성)
        if rng.random()<0.10: head="N"
        if rng.random()<0.08: homeless="N"
        if rng.random()<0.05: dup="Y"
        combined=ann+sp; debt=int(rng.integers(0,combined*2+2000)); na=int(rng.integers(0,40000))
        pr.append([uid,age,2026-age,int(rng.integers(1,13)),rng.choice(["남","여"]),emp,mar,my,nb,head,homeless,
                   ann,sp,combined,credit,debt,na,dup])
        if size=="small":   mk=int(rng.integers(3500,12000))
        elif size=="large": mk=int(rng.integers(30000,90000)) if cap else int(rng.integers(18000,42000))
        else:               mk=int(rng.integers(10000,30000)) if cap else int(rng.integers(7000,20000))
        dp=int(mk*rng.uniform(0.50,0.92)); sr=int(mk*rng.uniform(0,0.20))
        area=round(float(rng.uniform(16,84)),1) if rng.random()<0.75 else round(float(rng.uniform(85,140)),1)
        rg,dt=(CAP if cap else NON)[int(rng.integers(0,len(CAP if cap else NON)))]
        gr.append([uid,rg,dt,"Y" if cap else "N",rng.choice(PT),area,dp,mk,sr,int(dp*rng.uniform(0.65,0.80)),
                   round(dp/mk*100,1),round((sr+dp)/mk*100,1),f"2026-{int(rng.integers(7,13)):02d}-{int(rng.integers(1,28)):02d}",
                   rng.choice(RP),rng.choice(IR),"은행" if rng.random()<0.7 else "2금융권",dsr,info])
    return pd.DataFrame(pr,columns=prof0.columns), pd.DataFrame(gr,columns=gate0.columns)

In [ ]:
# ===== [Cell 1] 데이터 로드 =====
prof = pd.read_csv(P_PROF)   # 고객 개인·재무
gate = pd.read_csv(P_GATE)   # 임차주택·계약
cat  = pd.read_csv(P_CAT)    # 정부지원+시장 상품 카탈로그
fss  = pd.read_csv(P_FSS)    # 금감원 API 실제 상품(은행/저축은행/보험)

if N_NEW > 0:                                    # 통과 잘 되는 고객 붙이기
    pnew, gnew = grow_customers(prof, gate, N_NEW, SEED)
    prof = pd.concat([prof, pnew], ignore_index=True)
    gate = pd.concat([gate, gnew], ignore_index=True)
    # 증식된 입력 데이터도 파일로 저장(추천 결과와 동일한 입력)
    prof.to_csv("jeonse_customer_profiles_expanded.csv", index=False, encoding="utf-8-sig")
    gate.to_csv("jeonse_property_gate_expanded.csv", index=False, encoding="utf-8-sig")

# 함정 1: 이 데이터는 금액이 전부 '만원' 단위라 단위는 통일돼 있음.
#         대신 DSR·금리·보증금비율은 '%' 단위 → 금액과 절대 그냥 더하면 안 됨. 각각 정규화해서 사용.
cust = prof.merge(gate, on="user_id", how="inner")   # 고객 = 재무 + 계약

# 상품 유니버스 = 정부지원 A1~A5(catalog) + 시장(fss)
# catalog의 B 상품은 fss 데이터와 동일한 은행 상품 → catalog에선 A 상품만 사용해 중복 방지
cat_gov = cat[cat["product_id"].str.startswith("A")].copy()
cat_gov["source"] = "정부지원"

_lim = pd.to_numeric(fss["loan_lmt_man_won"], errors="coerce").fillna(0)
fss2 = fss[fss["base_rate_min"].notna()].assign(
    product_id=fss["fin_prdt_cd"].astype(str)+"@"+fss["fin_co_no"].astype(str),
    product_name=fss["kor_co_nm"].str.strip()+" "+fss["fin_prdt_nm"].str.replace("\n"," ",regex=False).str.strip(),
    deposit_limit_capital=99999, deposit_limit_non_capital=99999,
    max_loan_capital=_lim.where(_lim>0,99999), max_loan_non_capital=_lim.where(_lim>0,99999),
    source=fss["financial_sector"])[["product_id","product_name","source","age_min","age_max",
    "income_limit_single","income_limit_couple","net_asset_limit","credit_score_min",
    "deposit_limit_capital","deposit_limit_non_capital","max_loan_capital","max_loan_non_capital",
    "base_rate_min","base_rate_max","max_area_sqm","guarantee_agency","dsr_limit_pct",
    "marriage_within_7y_required","newborn_within_2y_required","youth_only","allowed_property_types"]]
prod = pd.concat([cat_gov, fss2], ignore_index=True)

print("cust", cust.shape, "| prod", prod.shape,
      "| 상품(정부", (prod.source=='정부지원').sum(), "/ 시장", (prod.source!='정부지원').sum(), ")")
print(cust.head(3)[["user_id","age","combined_annual_income","credit_score_kcb",
                    "deposit_amount","dsr_estimated_pct","is_homeless"]].to_string(index=False))

In [ ]:
# ===== [Cell 2] 정규화 도구 (적금 실습과 동일) =====
def minmax(s):       # 클수록 좋은 항목 → 0~100점
    s = s.astype(float)
    return 100*(s-s.min())/(s.max()-s.min()) if s.max()>s.min() else pd.Series(50.0, index=s.index)

def reverse(s):      # 작을수록 좋은 항목(부담·금리·위험) → 0~100점
    return 100 - minmax(s)

# 확인: 상품 금리를 reverse하면 금리 낮은 상품이 높은 점수
print(reverse(prod["base_rate_min"]).round(1).head(3).tolist())

In [ ]:
# ===== [Cell 3] 고객 x 상품 조합 + JeonseLoanGate (12항목 곱) =====
df = cust.merge(prod, how="cross")               # 고객 x 상품
df["income_cap"]  = np.where(df["marital_status"]=="기혼", df["income_limit_couple"], df["income_limit_single"])
df["deposit_cap"] = np.where(df["is_capital_area"]=="Y", df["deposit_limit_capital"], df["deposit_limit_non_capital"])
df["max_loan"]    = np.where(df["is_capital_area"]=="Y", df["max_loan_capital"], df["max_loan_non_capital"])

# md Gate 표 기준: 무주택/세대주는 정부지원·은행 구분 없이 모든 상품에 동일하게 적용
df["gate"] = (
    df["age"].between(df["age_min"], df["age_max"]) &
    (df["is_homeless"]=="Y") &
    (df["is_household_head"]=="Y") &
    (df["combined_annual_income"]<=df["income_cap"]) & (df["net_asset"]<=df["net_asset_limit"]) &
    (df["credit_score_kcb"]>=df["credit_score_min"]) &
    (df["dsr_estimated_pct"]<=df["dsr_limit_pct"]) &
    (df["deposit_amount"]<=df["deposit_cap"]) &
    df.apply(lambda r: r["property_type"] in str(r["allowed_property_types"]).split("|"), axis=1) &
    (df["exclusive_area_sqm"]<=df["max_area_sqm"]) &
    (df["has_existing_jeonse_loan"]=="N") & (df["info_completeness"]=="Y") &
    np.where(df["marriage_within_7y_required"]=="Y",(df["marital_status"]=="기혼")&(df["marriage_years"]<=7),True) &
    np.where(df["newborn_within_2y_required"]=="Y", df["has_newborn_within_2y"]=="Y", True) &
    np.where(df["youth_only"]=="Y", df["age"].between(19,34), True)
).astype(int)

print("Gate 통과 조합:", int(df["gate"].sum()), "/", len(df),
      "| 1개 이상 통과 고객:", df[df.gate==1]["user_id"].nunique(), "/", cust["user_id"].nunique())

In [ ]:
# ===== [Cell 4] 세부점수 6종 + 유형별 가중치 → score =====
# 클수록 +
df["Approval"]  = 0.6*((df["credit_score_kcb"]-df["credit_score_min"])/(1000-df["credit_score_min"])).clip(0,1)*100 \
                + 0.4*((df["income_cap"]-df["combined_annual_income"])/df["income_cap"]).clip(0,1)*100
# 함정 2: 금리는 base_rate_min만 쓰면 금리범위 넓은 저축은행(3.07~17%)이 '최저금리'로 1등됨 → 중간값 사용
prod["rep_rate"] = (prod["base_rate_min"]+prod["base_rate_max"])/2
df["rep_rate"]   = df["product_id"].map(prod.set_index("product_id")["rep_rate"])
# 함정 3: 금리 정규화를 전체 상품 기준으로 하면 고객마다 후보군이 달라도 동일한 점수가 매겨짐
#         → md의 "후보군 내 정규화" 기준에 따라 Gate 통과 상품 내에서 고객별로 비교
_rng = (df[df.gate==1].groupby("user_id")["rep_rate"]
        .agg(rate_lo="min", rate_hi="max").reset_index())
df = df.merge(_rng, on="user_id", how="left")
df["Interest"] = np.where(
    (df["gate"]==1) & (df["rate_hi"] > df["rate_lo"]),
    ((df["rate_hi"] - df["rep_rate"]) / (df["rate_hi"] - df["rate_lo"]) * 100).clip(0, 100),
    np.where(df["gate"]==1, 50.0, 0.0)
)
df = df.drop(columns=["rate_lo", "rate_hi"])
df["Limit"]      = (df["max_loan"]/df["loan_amount_requested"]).clip(upper=1)*100           # 한도 커버율
df["Afford"]     = ((df["dsr_limit_pct"]-df["dsr_estimated_pct"])/df["dsr_limit_pct"]*100).clip(lower=0)  # DSR 여유
df["Guarantee"]  = df["guarantee_agency"].map({"HUG":100,"HF":80,"SGI":60}).fillna(70)      # 보증 안정성
df["Pref"]       = ((df["employment_type"]=="정규직").astype(int)+(df["credit_score_kcb"]>=800).astype(int)
                    +(df["net_asset"]>0).astype(int))/3*100                                 # 우대 근사
# 감점 (고객·계약 리스크)
df["Penalty"] = (np.where(df["dsr_estimated_pct"].between(30,40,inclusive="left"),10,0)
               + np.where(df["credit_score_kcb"].between(600,699),15,0)
               + np.where(df["deposit_to_market_ratio_pct"]>90,20,0)
               + np.where(df["senior_plus_deposit_to_market_pct"]>80,10,0))

# 고객 유형 판별 → 여러 유형 매칭 시 '실패비용 큰 보호형' 우선 1개 채택
W = {"기본형":(.25,.25,.20,.15,.10,.05),"금리민감형":(.20,.38,.18,.14,.05,.05),
     "한도부족형":(.20,.15,.38,.15,.07,.05),"신용취약형":(.40,.20,.18,.12,.05,.05),
     "청년무주택형":(.20,.35,.20,.10,.10,.05),"신혼·신생아형":(.22,.30,.25,.13,.05,.05),
     "상환위험형":(.25,.18,.15,.32,.05,.05)}
PRI = ["신용취약형","상환위험형","한도부족형","신혼·신생아형","청년무주택형","금리민감형","기본형"]
# 한도부족형: 시장(은행) 최대한도(수도권 5억/비수도권 3억) 기준 → 진짜 큰 대출만 해당(정부 캡으로 잡으면 대부분 걸림)
rcap = np.where(cust["is_capital_area"]=="Y",50000,30000)
flag = pd.DataFrame({"user_id":cust["user_id"],
    "신용취약형":cust["credit_score_kcb"].between(600,699),
    "상환위험형":cust["dsr_estimated_pct"]>=35,
    "한도부족형":cust["loan_amount_requested"]>0.9*rcap,
    "신혼·신생아형":((cust["marital_status"]=="기혼")&(cust["marriage_years"]<=7))|(cust["has_newborn_within_2y"]=="Y"),
    "청년무주택형":cust["age"].between(19,34)&(cust["is_homeless"]=="Y")&(cust["is_household_head"]=="Y"),
    "금리민감형":cust["combined_annual_income"]<=4000,"기본형":True})
flag["profile"]       = flag.apply(lambda r:[t for t in PRI if r[t]][0], axis=1)
flag["matched_types"] = flag.apply(lambda r:",".join([t for t in PRI if r[t]]), axis=1)
df = df.merge(flag[["user_id","profile","matched_types"]], on="user_id")
w  = df["profile"].map(W).apply(pd.Series); w.columns=["wA","wI","wL","wF","wG","wP"]

df["score"] = df["gate"] * (
      w["wA"]*df["Approval"]  + w["wI"]*df["Interest"] + w["wL"]*df["Limit"]
    + w["wF"]*df["Afford"]    + w["wG"]*df["Guarantee"]+ w["wP"]*df["Pref"]
    - df["Penalty"]).clip(lower=0)

print(df[df.gate==1]["score"].describe().round(1)[["min","mean","max"]].to_dict())

In [ ]:
# ===== [Cell 5] 고객당 TOP 5  (정부상품 강제노출 + 쏠림 분산) =====
pas = df[df.gate==1].copy()

# 쏠림 분산 ①: 같은 금융회사 상품은 최고점 1개만 (케이뱅크 일반+고정 중복 방지)
pas["company"] = pas["product_id"].str.split("@").str[-1]     # fss는 fin_co_no, 정부상품은 A1~A6
pas = pas.sort_values("score", ascending=False).drop_duplicates(["user_id","company"])

# 쏠림 분산 ②(정부상품 강제노출): 자격(연령·혼인·출산·소득)되고 Gate 통과한 정부상품은 TOP5에 반드시 포함
pas["forced"] = (
    (pas["age"].between(19,34) & pas["product_id"].isin(["A2","A5"])) |                 # 청년→A2·A5
    ((pas["marital_status"]=="기혼") & (pas["marriage_years"]<=7) & (pas["product_id"]=="A3")) |  # 신혼→A3
    ((pas["has_newborn_within_2y"]=="Y") & (pas["product_id"]=="A4")) |                 # 신생아→A4
    ((pas["combined_annual_income"]<=5000) & (pas["product_id"]=="A1"))                 # 저소득→A1
).astype(int)

# 강제상품(forced) 먼저, 그다음 점수순 → 고객당 상위 5개
pas = pas.sort_values(["user_id","forced","score"], ascending=[True,False,False])
top5 = pas.groupby("user_id", as_index=False).head(TOP_N)
top5 = top5.sort_values(["user_id","score"], ascending=[True,False])

# sanity check: 리스크 감점 있는 조합이 없는 조합보다 점수가 낮아야 정상
print("감점 있음 평균:", round(pas[pas.Penalty>0].score.mean(),1),
      "| 감점 없음 평균:", round(pas[pas.Penalty==0].score.mean(),1))
# 쏠림 개선 확인: 기금(버팀목 A1~A5)이 TOP5에 든 고객 수, 최다 추천 상품 비중(낮을수록 분산 잘 됨)
print("기금(버팀목) TOP5 포함 고객:", top5[top5["product_id"].isin(["A1","A2","A3","A4","A5"])]["user_id"].nunique(),
      "/", top5["user_id"].nunique())
print("최다 추천 상품 비중: {:.0%} (분산 전에는 한 상품이 90%+ 였음)".format(
      top5["product_name"].value_counts(normalize=True).iloc[0]))

In [ ]:
# ===== [Cell 6] action / reason  (상품별 구체적 추천 사유) =====
def action(s):
    if s>=70: return "메인 추천"
    if s>=55: return "리스트 노출"
    if s>=40: return "조건 확인 유도"
    return "추천 보류"

def make_reason(r):                     # 이 상품이 이 고객에게 추천된 핵심 근거 최대 3개
    t = []
    if r["source"]=="정부지원":      t.append(f"정부지원 저금리(연 {r['rep_rate']:.1f}%) 자격 충족")
    if r["Interest"]>=85:            t.append("후보 중 최저금리대")
    if r["Limit"]>=100:              t.append("필요 보증금 전액 커버")
    elif r["Limit"]<70:              t.append("한도 부족 — 보증금 일부만 커버")
    if r["Afford"]>=80:              t.append("DSR 여유 충분")
    if r["Approval"]>=80:            t.append("승인 가능성 높음")
    if r["Penalty"]>0:               t.append(f"리스크 감점 −{int(r['Penalty'])}(신용/DSR/깡통전세)")
    return " · ".join(t[:3]) if t else "자격 충족, 표준 추천"

top5["action"] = top5["score"].apply(action)
top5["reason"] = top5.apply(make_reason, axis=1)
print(top5["action"].value_counts().to_dict())

# 샘플: 청년·신혼·신생아 기금상품이 실제로 TOP5에 강제노출된 고객 1명 보여주기
gita = top5[top5["product_id"].isin(["A2","A3","A4"])]
demo = (gita if len(gita) else top5[top5.source=='정부지원'])["user_id"].iloc[0]
d = top5[top5.user_id==demo]
print(f"\n[{demo}] 프로파일={d.iloc[0]['profile']} (매칭:{d.iloc[0]['matched_types']})")
print(d[["product_name","source","rep_rate","score","action","reason"]].round(1).to_string(index=False))

In [ ]:
# ===== [Cell 7] 저장 =====
out = top5[["user_id","profile","matched_types","product_name","source","guarantee_agency",
            "rep_rate","max_loan","score","action","reason"]].copy()
out["score"]=out["score"].round(1); out["rep_rate"]=out["rep_rate"].round(2)
out = out.rename(columns={"rep_rate":"rate_%","max_loan":"limit_만원"})

# 통과 상품이 없는 고객 → 구체적 '제외사유' 계산해서 남김
def why_excluded(r):                    # 어느 게이트에서 걸렸는지 (은행·기금 각각)
    t = []
    if r["has_existing_jeonse_loan"]=="Y":      t.append("기존 전세대출 보유(중복 불가)")
    if r["info_completeness"]=="N":             t.append("필수정보 미비")
    if r["credit_score_kcb"]<600:               t.append(f"신용점수 과소({int(r['credit_score_kcb'])})")
    elif r["credit_score_kcb"]<650:             t.append(f"신용 650 미만({int(r['credit_score_kcb'])})—은행 불가")
    if r["dsr_estimated_pct"]>50:               t.append(f"DSR 과다({r['dsr_estimated_pct']}%)")
    elif r["dsr_estimated_pct"]>40:             t.append(f"DSR 40% 초과({r['dsr_estimated_pct']}%)—은행 불가")
    if r["is_homeless"]=="N":                   t.append("무주택 아님—기금 불가")
    if r["is_household_head"]=="N":             t.append("세대주 아님—기금 불가")
    if r["net_asset"]>34500:                    t.append("순자산 3.45억 초과—기금 불가")
    return " / ".join(t) if t else "자격 요건 미충족"

nop = sorted(set(cust["user_id"])-set(pas["user_id"]))
if nop:
    ex = cust[cust.user_id.isin(nop)].merge(flag[["user_id","profile","matched_types"]], on="user_id")
    ex["reason"] = ex.apply(why_excluded, axis=1)
    ex["action"] = "추천 불가"; ex["score"] = 0.0
    for c in ["product_name","source","guarantee_agency","rate_%","limit_만원"]: ex[c]=np.nan
    out = pd.concat([out, ex[out.columns]], ignore_index=True)
out = out.sort_values(["user_id","score"], ascending=[True,False])

out.to_csv("41_jeonse_recommendation.csv", index=False, encoding="utf-8-sig")
print("저장 완료:", out.shape, "| 추천 고객:", out[out.action!="추천 불가"]["user_id"].nunique(),
      "| 추천 불가:", len(nop))
print("\n[추천 사유 예시]")
print(out[out.action!="추천 불가"].head(3)[["user_id","product_name","action","reason"]].to_string(index=False))
print("\n[제외 사유 예시]")
print(out[out.action=="추천 불가"].head(5)[["user_id","profile","reason"]].to_string(index=False))

try:
    from google.colab import files
    files.download("41_jeonse_recommendation.csv")
except ImportError:
    pass